In [1]:
import numpy as np
import random
import import_ipynb
from GR4J import compute_Q
from Metric_Calculation import compute_KGE, compute_NSE, compute_PBIAS
import pandas as pd

30.7 ms ± 1.11 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
5.684341886080802e-14
[0.67712195 0.62997907 0.58832275 ... 0.60016149 0.58716065 0.92408118]


In [2]:
csv_path = r"D:\Claude\flood_hydrology_modeling\data\processed\prec_PET_sf.csv"
out_path = r"D:\Claude\flood_hydrology_modeling\data\processed\GR4J.csv"
df = pd.read_csv(csv_path)
P = df['prec_AGCD']
E = df['PET_morton']
obs = df['sf_mmd'].to_numpy(dtype=np.float64)   # keep NaN as-is, don't fill
warmup = 730  # 2-year warm-up (2*365)
x1, x2, x3, x4 = [350, 0, 90, 1.7]
x = np.array((x1,x2,x3,x4))

In [13]:
# Objective: minimize -KGE - the SAME objective as notebook 04, so this PSO run is
# directly comparable to RS-SPSO. 730-day warm-up discarded, NaN observations masked.
def score_kge(sim, obs, warmup):
    s = sim[warmup:]
    o = obs[warmup:]
    mask = ~np.isnan(o)
    return compute_KGE(o[mask], s[mask])[0]     # [0] = the KGE scalar

def objective_function(x):
    global P, E, obs, warmup
    sim = compute_Q(P, E, x)
    kge = score_kge(sim, obs, warmup)
    if not np.isfinite(kge):
        return 1e6           # push optimizer away from bad params
    return -kge

In [14]:
class Particle:
    def __init__(self, bounds):
        self.num_dimensions = len(bounds)

        #Initialize position randomly within bounds
        self.position = np.array([random.uniform(bounds[d][0], bounds[d][1]) for d in range(self.num_dimensions)])

        #Initialize velocity randomly between a reasonable fraction of the bounds
        self.velocity = np.array([random.uniform(-abs(bounds[d][1] - bounds[d][0])/10, abs(bounds[d][1] - bounds[d][0])/10) for d in range(self.num_dimensions)])

        #Initialize personal best to the starting configuration
        self.best_position = np.copy(self.position)
        self.best_score = objective_function(self.position)

    def update_position(self, bounds):
        #Update position vector
        self.position += self.velocity

        #keep position within problem boundaries(clamping)
        for d in range(self.num_dimensions):
            if self.position[d] < bounds[d][0]:
                self.position[d] = bounds[d][0]
            if self.position[d] > bounds[d][1]:
                self.position[d] = bounds[d][1]

# Main PSO Algorithm function
def particle_swarm_optimization(
    objective_func,
    bounds,
    num_particles =30,
    max_iterations=50,
    w=0.5,
    c1=1.5,
    c2=1.5,
    ):
    #create swarm
    swarm = [Particle(bounds) for _ in range(num_particles)]

    #Initialize global best and best position
    global_best_score = float("inf")
    global_best_position = np.zeros(len(bounds))

    #Main optimization loop
    for iteration in range(max_iterations):
        for particle in swarm:
            #Evaluate current fitness
            current_score = objective_func(particle.position)

            #Update personal best if current position is better
            if current_score < particle.best_score:
                particle.best_score = current_score
                particle.best_position = np.copy(particle.position)

            #Update global best if personal best is better
            if current_score < global_best_score:
                global_best_score = current_score
                global_best_position = np.copy(particle. position)
        #Update velocities and positions for the next step
        for particle in swarm:
            #Generate random coefficients for this step
            r1 = random.random()
            r2 = random.random()
            #Calculate new velocity
            cognitive_velocity = c1 * r1 * (particle.best_position - particle.position)
            social_velocity = c2 * r2 * (global_best_position - particle.position)
            particle.velocity = (w*particle.velocity + cognitive_velocity + social_velocity)

            #Move particle
            particle.update_position(bounds)
        #Print progress every 10 iterations
        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(f" Iteration: {iteration+1:02d}/{max_iterations} | Best Score: {global_best_score:.6f}")
    return global_best_position, global_best_score


In [15]:
#Run the optimization
if __name__ == "__main__":
    search_bounds = [(1, 1500), (-5, 5), (1, 500), (0.5, 4)]   # x1, x2, x3, x4
    print("Starting Particle Swarm Optimization.....
")
    best_pos, best_val = particle_swarm_optimization(objective_function, search_bounds)
    print("Optimization Complete>>
")
    print(f"Best Position Found: [{best_pos}]")
    print(f"Best KGE: {-best_val:.4f}")

Starting Particle Swarm Optimization.....

 Iteration: 01/50 | Best Score: -38.011182
 Iteration: 10/50 | Best Score: -362.888123
 Iteration: 20/50 | Best Score: -362.888123
 Iteration: 30/50 | Best Score: -362.888123
 Iteration: 40/50 | Best Score: -362.888123
 Iteration: 50/50 | Best Score: -362.888123
Optimization Complete>>

Best Position Found: [[1. 5. 1. 4.]]
Target Global Minimum (0,0): [0.000000, 0.000000]
Objective Function Value: -3.628881e+02


In [16]:
print(best_pos)

[1. 5. 1. 4.]


In [17]:
sim = compute_Q(P, E, best_pos)
o = obs[730:]                # observed, after 2-yr warm-up
s = sim[730:]                # simulated, same window
mask = ~np.isnan(o)          # True where obs is a real number, False on the 7 NaN days
o, s = o[mask], s[mask]      # keep only valid days - and keep them aligned
print(compute_KGE(o, s))     # [KGE, r, alpha, beta]
print("NSE   =", compute_NSE(o, s))
print("PBIAS =", compute_PBIAS(o, s))

[-2.6701334221280644, 0.5352450934705829, 1.2917245691129537, 4.628881231020678]
NSE   = -1.960466386341547
PBIAS = -362.8881231020694
